# Jira Ticket Priority NLP — Google Colab

Fine-tune **DistilBERT** to predict Jira ticket **Priority** from **Summary + Description**.

**Recommended:** Runtime → Change runtime type → **GPU** (T4 is enough).

Pipeline:
1. Install dependencies & clone repo
2. Configure sample size / epochs
3. Load data (demo, CSV upload, or ASF BSON)
4. Train & evaluate
5. Download model + plots

## 1. Setup

In [ ]:
!pip install -q pandas numpy scikit-learn imbalanced-learn \
    torch transformers datasets accelerate matplotlib seaborn pymongo tqdm regex

import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — training will be slow. Use Runtime → Change runtime type → GPU.")

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/gayanz/jira_priority_nlp.git"
PROJECT_DIR = Path("/content/jira_priority_nlp")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", PROJECT_DIR)

## 2. Configuration

Adjust these before training. For a quick smoke test on Colab, keep the defaults below.

In [ ]:
import re

# --- edit these ---
SAMPLE_SIZE = 500          # use 5000 for full study
BERT_EPOCHS = 1            # use 3-5 for full study
BERT_BATCH_SIZE = 16       # reduce to 8 if you hit OOM
ASF_MAX_SCAN_ISSUES = 250_000
# ------------------

config_path = PROJECT_DIR / "config.py"
text = config_path.read_text(encoding="utf-8")

replacements = {
    r"SAMPLE_SIZE = \d+": f"SAMPLE_SIZE = {SAMPLE_SIZE}",
    r"BERT_EPOCHS = \d+": f"BERT_EPOCHS = {BERT_EPOCHS}",
    r"BERT_BATCH_SIZE = \d+": f"BERT_BATCH_SIZE = {BERT_BATCH_SIZE}",
    r"ASF_MAX_SCAN_ISSUES = [\d_]+": f"ASF_MAX_SCAN_ISSUES = {ASF_MAX_SCAN_ISSUES}",
}
for pattern, value in replacements.items():
    text = re.sub(pattern, value, text)

config_path.write_text(text, encoding="utf-8")
print(f"Config updated: SAMPLE_SIZE={SAMPLE_SIZE}, BERT_EPOCHS={BERT_EPOCHS}, BERT_BATCH_SIZE={BERT_BATCH_SIZE}")

## 3. Data

Choose **one** option:

| Option | When to use |
|--------|-------------|
| **A — Demo** | Quick pipeline test (synthetic data) |
| **B — CSV upload** | You already have `jira_asf_5000.csv` |
| **C — ASF BSON** | Real data from [Zenodo 5665896](https://zenodo.org/records/5665896) |

For BSON (large file), prefer mounting Google Drive and copying `issues.bson.gz` into `data/`.

In [ ]:
DATA_MODE = "demo"  # "demo" | "csv" | "bson"
# For BSON: "download" (recommended) or "drive"
BSON_SOURCE = "download"
DRIVE_BSON = "/content/drive/MyDrive/issues.bson.gz"  # only if BSON_SOURCE == "drive"

data_dir = PROJECT_DIR / "data"
data_dir.mkdir(exist_ok=True)
sample_csv = data_dir / "jira_asf_5000.csv"
issues_bson = data_dir / "issues.bson.gz"

if DATA_MODE == "demo":
    if sample_csv.exists():
        sample_csv.unlink()
    print("Demo mode: train.py will generate synthetic data on first run.")

elif DATA_MODE == "csv":
    from google.colab import files
    uploaded = files.upload()  # upload jira_asf_5000.csv
    for name, content in uploaded.items():
        (data_dir / name).write_bytes(content)
    print(f"Saved to {data_dir / list(uploaded.keys())[0]}")

elif DATA_MODE == "bson":
    from asf_data import (
        ISSUES_BSON_GZ_EXPECTED,
        ZENODO_ISSUES_URL,
        build_asf_sample_csv,
        validate_gzip_bson,
    )
    from prepare_asf_data import PRIORITY_LABEL_MAP

    expected_bytes = ISSUES_BSON_GZ_EXPECTED["size"]
    print(f"Expected issues.bson.gz size: {expected_bytes:,} bytes (~729 MB)")

    if BSON_SOURCE == "download":
        if not issues_bson.exists() or issues_bson.stat().st_size != expected_bytes:
            if issues_bson.exists():
                issues_bson.unlink()
            print("Downloading from Zenodo (may take several minutes)...")
            !wget -c -O {issues_bson} {ZENODO_ISSUES_URL}
    elif BSON_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        import shutil
        shutil.copy(DRIVE_BSON, issues_bson)
    else:
        raise ValueError("BSON_SOURCE must be 'download' or 'drive'")

    validate_gzip_bson(
        issues_bson,
        expected_size=expected_bytes,
        expected_md5=ISSUES_BSON_GZ_EXPECTED["md5"],
    )
    build_asf_sample_csv(priority_label_map=PRIORITY_LABEL_MAP)
    print("Built stratified CSV from BSON.")

else:
    raise ValueError("DATA_MODE must be 'demo', 'csv', or 'bson'")

## 4. Train & evaluate

In [ ]:
!python train.py

## 5. View results

In [ ]:
from IPython.display import Image, display

output_dir = PROJECT_DIR / "output"
for pattern in ["confusion_matrix*.png", "pr_curves*.png"]:
    for img in sorted(output_dir.glob(pattern)):
        print(img.name)
        display(Image(filename=str(img)))

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/jira_priority_nlp_outputs.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", output_dir)
files.download(zip_path)

## Optional: predict on a new ticket

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from preprocess import clean_text

model_dir = PROJECT_DIR / "output" / "distilbert_final"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

id2label = model.config.id2label

def predict_priority(summary: str, description: str = "") -> str:
    text = clean_text(summary) + " " + clean_text(description)
    enc = tokenizer(text, max_length=256, padding=True, truncation=True, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred_id = int(logits.argmax(dim=-1).item())
    return id2label[pred_id]

# Example
predict_priority(
    "Production outage: API returning 500 errors",
    "All users unable to login since 09:00 UTC.",
)